In [1]:
import tensorflow as tf
import os
import glob
from collections import defaultdict
import pandas as pd

2025-04-18 05:53:34.766943: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-18 05:53:34.960029: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744952015.031743     597 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744952015.051923     597 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-18 05:53:35.220811: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [2]:
def parse_song_name(example):
    feature_description = {
        'labels': tf.io.VarLenFeature(tf.int64),
        'mel_spectrogram': tf.io.FixedLenFeature([], tf.string),
        'song_name': tf.io.FixedLenFeature([], tf.string),
        'segment_idx': tf.io.FixedLenFeature([], tf.int64),
        'total_segments': tf.io.FixedLenFeature([], tf.int64)
    }
    parsed = tf.io.parse_single_example(example, feature_description)
    return parsed['song_name']

In [3]:
segment_lengths = [3, 6, 10, 14, 20, 30]

# store song names for each dataset
song_sets = {}

In [5]:
# Process each dataset
for seg_len in segment_lengths:
    # test dataset directory for curr segment length
    dataset_dir = f"../creating_spectrogram_batches/tfrecord_dataset_{seg_len}s"
    test_dir = os.path.join(dataset_dir, "test")
    
    if not os.path.exists(test_dir):
        print(f"Warning: Test directory for {seg_len}s segments not found at {test_dir}")
        continue
    
    tfrecord_files = glob.glob(os.path.join(test_dir, "*.tfrecord"))
    
    if not tfrecord_files:
        print(f"Warning: No TFRecord files found in {test_dir}")
        continue
    
    raw_dataset = tf.data.TFRecordDataset(tfrecord_files)
    
    song_names = set()
    for raw_record in raw_dataset:
        song_name = parse_song_name(raw_record).numpy().decode('utf-8')
        song_names.add(song_name)
    
    # Store the set of song names
    song_sets[seg_len] = song_names
    print(f"Found {len(song_names)} unique songs in {seg_len}s dataset")

I0000 00:00:1744952099.888054     597 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5520 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9
2025-04-18 05:55:00.008272: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:370] TFRecordDataset `buffer_size` is unspecified, default to 262144
2025-04-18 05:55:07.300335: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Found 160 unique songs in 3s dataset


2025-04-18 05:55:10.945594: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Found 160 unique songs in 6s dataset
Found 160 unique songs in 10s dataset


2025-04-18 05:55:14.106065: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Found 160 unique songs in 14s dataset
Found 160 unique songs in 20s dataset
Found 160 unique songs in 30s dataset


In [6]:
if len(song_sets) >= 2:
    # Create a comparison matrix
    comparison_data = defaultdict(dict)
    
    for seg_len1 in song_sets:
        for seg_len2 in song_sets:
            if seg_len1 == seg_len2:
                # Diagonal - show the number of songs
                comparison_data[f"{seg_len1}s"][f"{seg_len2}s"] = len(song_sets[seg_len1])
            else:
                # Off-diagonal - show the overlap
                overlap = len(song_sets[seg_len1].intersection(song_sets[seg_len2]))
                comparison_data[f"{seg_len1}s"][f"{seg_len2}s"] = overlap
    
    comparison_df = pd.DataFrame(comparison_data)
    print("\nComparison Matrix (values show number of overlapping songs):")
    print(comparison_df)
    
    print("\nDifferences between datasets:")
    for seg_len1 in song_sets:
        for seg_len2 in song_sets:
            if seg_len1 < seg_len2:  # Avoid duplicating comparisons
                diff1 = song_sets[seg_len1] - song_sets[seg_len2]
                diff2 = song_sets[seg_len2] - song_sets[seg_len1]
                
                if diff1 or diff2:
                    print(f"\nDifference between {seg_len1}s and {seg_len2}s datasets:")
                    if diff1:
                        print(f"Songs in {seg_len1}s but not in {seg_len2}s: {len(diff1)} songs")
                        # Print a few examples if there are differences
                        if len(diff1) > 0:
                            print(f"Examples: {list(diff1)[:5]}")
                    if diff2:
                        print(f"Songs in {seg_len2}s but not in {seg_len1}s: {len(diff2)} songs")
                        if len(diff2) > 0:
                            print(f"Examples: {list(diff2)[:5]}")
                else:
                    print(f"{seg_len1}s and {seg_len2}s datasets have identical song sets")
else:
    print("Not enough datasets found to compare")


Comparison Matrix (values show number of overlapping songs):
      3s   6s  10s  14s  20s  30s
3s   160  160  160  160  160  160
6s   160  160  160  160  160  160
10s  160  160  160  160  160  160
14s  160  160  160  160  160  160
20s  160  160  160  160  160  160
30s  160  160  160  160  160  160

Differences between datasets:
3s and 6s datasets have identical song sets
3s and 10s datasets have identical song sets
3s and 14s datasets have identical song sets
3s and 20s datasets have identical song sets
3s and 30s datasets have identical song sets
6s and 10s datasets have identical song sets
6s and 14s datasets have identical song sets
6s and 20s datasets have identical song sets
6s and 30s datasets have identical song sets
10s and 14s datasets have identical song sets
10s and 20s datasets have identical song sets
10s and 30s datasets have identical song sets
14s and 20s datasets have identical song sets
14s and 30s datasets have identical song sets
20s and 30s datasets have identical